# 1.	Building a Simple LCEL Chain: Create a minimal LCEL script that accepts a fixed instruction (e.g., “Summarize this text: …”), passes it to an LLM, and prints the result; verify end-to-end execution.!

In [1]:
pip install langchain-core langchain-openai


Note: you may need to restart the kernel to use updated packages.


In [31]:
pip install langchain-core langchain-groq


Note: you may need to restart the kernel to use updated packages.


In [33]:
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
# Replace langchain_openai with langchain_groq
from langchain_groq import ChatGroq

def main():
    # 1. Environment Verification
    # Changed verification to check for the GROQ_API_KEY
    if "GROQ_API_KEY" not in os.environ:
        print("[WARNING]: GROQ_API_KEY environment variable is not set!")
        print("Please set it in your terminal: export GROQ_API_KEY='your-api-key'")

    print("--- Lab Step 1: Initializing LCEL Components ---")

    # 2. Component A: The Prompt Template
    # This prepares the raw input text and prepends the fixed instruction.
    instruction = "Summarize the following text in one clear, concise sentence:"
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful and concise assistant."),
        ("user", f"{instruction}\n\n{{text}}")
    ])
    print("[OK] Prompt Template initialized with fixed instruction.")

    # 3. Component B: The Chat Model
    # The brain of our pipeline. We set temperature to 0 for consistent, deterministic outputs.
    # Replaced ChatOpenAI with ChatGroq and set a supported model (llama-3.3-70b-specdec)
    model = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
    print("[OK] Chat Model (openai/gpt-oss-120b via Groq) initialized.")

    # 4. Component C: The Output Parser
    # By default, ChatGroq returns an AIMessage object. This parser extracts the raw text content.
    parser = StrOutputParser()
    print("[OK] String Output Parser initialized.")

    # 5. Component D: Composition (The LCEL Pipe Syntax)
    # The '|' operator chains our components together. 
    # Data is automatically fed from the prompt to the model, and then to the parser.
    chain = prompt | model | parser
    print("[OK] LCEL Chain successfully composed: prompt | model | parser\n")

    # 6. Verification & Execution
    sample_text = (
        "LangChain Expression Language (LCEL) is a declarative way to easily chain "
        "LangChain components together. LCEL was designed from day one to support "
        "streaming, async, batching, and parallel execution of components, making "
        "it highly optimized for moving chains from prototype to production."
    )
    
    print("--- Lab Step 2: Executing the Chain ---")
    print(f"Input Text:\n\"{sample_text}\"\n")
    print("Sending request to Groq LLM...")

    try:
        # invoke() triggers the chain end-to-end
        result = chain.invoke({"text": sample_text})
        
        print("\n--- Lab Step 3: Result Verification ---")
        print(f"Summarized Output:\n{result}")
        print("\n[SUCCESS] End-to-end LCEL execution verified successfully!")
        
    except Exception as e:
        print(f"\n[ERROR] Execution failed: {e}")

if __name__ == "__main__":
    main()


--- Lab Step 1: Initializing LCEL Components ---
[OK] Prompt Template initialized with fixed instruction.
[OK] Chat Model (openai/gpt-oss-120b via Groq) initialized.
[OK] String Output Parser initialized.
[OK] LCEL Chain successfully composed: prompt | model | parser

--- Lab Step 2: Executing the Chain ---
Input Text:
"LangChain Expression Language (LCEL) is a declarative way to easily chain LangChain components together. LCEL was designed from day one to support streaming, async, batching, and parallel execution of components, making it highly optimized for moving chains from prototype to production."

Sending request to Groq LLM...

--- Lab Step 3: Result Verification ---
Summarized Output:
LangChain Expression Language (LCEL) is a declarative framework that seamlessly chains LangChain components while natively supporting streaming, async, batching, and parallel execution for efficient production deployment.

[SUCCESS] End-to-end LCEL execution verified successfully!


# Basic Data Indexing for RAG: Load a small collection of documents; split into uniform chunks (e.g., 200 tokens); generate embeddings for each chunk; store them in an in-memory vector store; inspect for consistency.

In [35]:
pip install langchain-core langchain-text-splitters langchain-groq langchain-community sentence-transformers


  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl.metadata (4.2 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 913.3 kB/s eta 0:00:00 kB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 2.0 MB/s eta 0:00:000m eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 2.0 MB/s eta 0:00:001.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 2.0 MB/s eta 0:00:00 MB/s eta 

In [3]:
pip install langchain_groq


Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install langchain-huggingface


Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Replace langchain_openai with langchain_groq
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# 0. Environment Verification
if "GROQ_API_KEY" not in os.environ:
    print("[WARNING]: GROQ_API_KEY environment variable is not set!")
    print("Please set it in your terminal: export GROQ_API_KEY='your-groq-api-key'")

# 1. Load Documents
documents = [
    Document(page_content="RAG prevents hallucination...", metadata={"source": "rag.txt"})
]

# 2. Chunk Documents (200 tokens/characters) [1]
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = text_splitter.split_documents(documents)

# 3 & 4. Embed & Store In-Memory [1]
# We replace OpenAIEmbeddings with GroqEmbeddings using Groq's standard embedding model
#embeddings = GroqEmbeddings(model="nomic-embed-text-v1.5")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)

# 5. Query and Inspect
# Search the store to verify retrieval consistency [1]
results = vector_store.similarity_search("How to prevent hallucinations?", k=1)

# Print verification results
for doc in results:
    print(f"Retrieved Content: {doc.page_content}")
    print(f"Source Metadata: {doc.metadata}")



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/homebrew/Caskroom/miniforge/base/envs/myenv/lib/python3.12/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/homebrew/Caskroom/miniforge/base/envs/myenv/lib/python3.12/runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "/opt/homebrew/Caskroom/miniforge/base/envs/myenv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/homebrew/Caskroom/miniforge/base/envs/myenv/lib/python3.12

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retrieved Content: RAG prevents hallucination...
Source Metadata: {'source': 'rag.txt'}


# 3.	Constructing & Running a Basic RAG Chain: Build a pipeline that:
        a.	Receives a user query
        b.	Retrieves the top-k relevant chunks
        c.	Constructs a combined prompt with context + query
        d.	Send it to the LLM
        e.	Returns the answer
Test with sample queries and compare factual accuracy against a prompt without retrieval.


In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# 1. Initialize the Embeddings and Vector Store
# Groq uses the fast nomic embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 1})

# 2. Define the RAG System Prompt
# Note the constraint instructing the LLM to restrict its knowledge base
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using ONLY the provided context. If you do not know, say so."),
    ("user", "Context:\n{context}\n\nQuestion: {question}")
])

# Helper to join text chunks
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 3. Assemble the LCEL Chain
# Let's break down how data passes through this dictionary structure:
rag_chain = (
    {
        "context": retriever | format_docs,   # Step 3b: Retrieve & Format chunks 
        "question": RunnablePassthrough()     # Step 3a: Keep raw user query 
    }
    | rag_prompt                              # Step 3c: Construct combined prompt 
    | ChatGroq(model="openai/gpt-oss-120b")   # Step 3d: Send to LLM 
    | StrOutputParser()                       # Step 3e: Parse final answer 
)

query = "Where is the 2026 Galactic AI Summit being held?"

# 4. Invoke the RAG chain
rag_response = rag_chain.invoke(query)
print(rag_response)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

I don’t know.
